# Linked Scatter, Clustergram, and Landscape — Xenium pancreas

Explore **122,678 cells** from the same human pancreas dataset used in the islet tutorial. The three panels share cell IDs and gene choices:

- **Clustergram gene click:** choose Scatter's Y gene and color Landscape by that gene.
- **Clustergram cluster click / dendrogram selection:** select those cells in Scatter and highlight them in Landscape.
- **Scatter GATE or SKTCH:** select individual cells and highlight the same cells in Landscape.
- **Scatter LABEL:** add a cell annotation and color, then retain it while changing genes, UMAP, or spatial view.
- **CELL category bars:** reflect the cells currently in view; Shift-click categories to select multiple populations. The attribute dropdown also offers numerical cell metadata.

The Clustergram remains a cluster-level expression overview; gating does not silently re-cluster it. A live kernel is required for new gene axes and annotations. The panels sit side by side in a horizontally scrollable row on smaller screens.

Use the current development checkout (`pip install -e .` and `npm run build` from the repository root). The first data load downloads about 110 MB and caches it locally. No additional Scanpy preprocessing is needed.

In [ ]:
from pathlib import Path
import os
from html import escape

# Use the bundle built from this branch, before importing Celldega.
os.environ["CELLDEGA_LOCAL_ESM"] = "1"

import anndata as ad
import h5py
import numpy as np
import pandas as pd
import requests
from scipy import sparse
from IPython.display import display
from ipywidgets import HBox, VBox, HTML, Layout, jsdlink

import celldega as dega

## Load the existing pancreas data

The processed AnnData comes from `broadinstitute/Celldega_Supporting_Data`, also used in `Gradient_Neighborhood_Pancreas_Islets.ipynb`. It has UMAP, spatial coordinates, and Leiden labels. Its `X` is already log-normalized, so we also load the matching small 10x counts file for a genuine **linear ↔ log1p** display. The original processed `X` is preserved.

Set `CELLDEGA_PANCREAS_DATA_DIR` to an existing cache if desired. Downloads use a temporary file and only replace the cache entry after a complete response.

In [ ]:
project_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()),
    Path.cwd(),
)
cache_dir = Path(os.environ.get(
    "CELLDEGA_PANCREAS_DATA_DIR",
    str(project_root / "notebooks/data/celldega_supporting_data"),
))
cache_dir.mkdir(parents=True, exist_ok=True)

def cached_download(filename, url):
    target = cache_dir / filename
    if target.is_file():
        return target
    temporary = target.with_suffix(target.suffix + ".part")
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with temporary.open("wb") as handle:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    handle.write(chunk)
    temporary.replace(target)
    return target

adata_path = cached_download(
    "Xenium_V1_human_Pancreas_FFPE_outs.h5ad",
    "https://huggingface.co/datasets/broadinstitute/Celldega_Supporting_Data/resolve/main/"
    "Xenium_V1_human_Pancreas_FFPE_outs.h5ad",
)
counts_path = cached_download(
    "cell_feature_matrix.h5",
    "https://raw.githubusercontent.com/broadinstitute/"
    "celldega_Xenium_human_Pancreas_FFPE/main/cell_feature_matrix.h5",
)
adata = ad.read_h5ad(adata_path)
# The saved AnnData uses integer-like obs names; Landscape uses Xenium cell IDs.
adata.obs_names = pd.Index(adata.obs["cell_id"].astype(str), name="cell_id")
assert adata.obs_names.is_unique and adata.var_names.is_unique

with h5py.File(counts_path, "r") as handle:
    matrix = handle["matrix"]
    raw_counts = sparse.csc_matrix(
        (matrix["data"][:], matrix["indices"][:], matrix["indptr"][:]),
        shape=tuple(matrix["shape"][:]),
    ).T.tocsr()
    raw_cells = pd.Index(matrix["barcodes"].asstr()[:])
    raw_genes = pd.Index(matrix["features"]["name"].asstr()[:])
row_index = raw_cells.get_indexer(adata.obs_names)
col_index = raw_genes.get_indexer(adata.var_names)
assert (row_index >= 0).all() and (col_index >= 0).all(), "Raw counts must match AnnData IDs."
adata.layers["counts"] = raw_counts[row_index][:, col_index].copy()
del raw_counts
adata.obs["manual_gate"] = pd.Categorical(["Unlabeled"] * adata.n_obs)
adata.uns["manual_gate_colors"] = ["#a0a0a0"]
print(f"{adata.n_obs:,} cells × {adata.n_vars:,} genes; counts, UMAP, spatial, and Leiden ready")

## Build a small cluster-level expression matrix

Aggregate the sparse expression matrix **before** constructing the Clustergram. This keeps the heatmap compact and avoids making a dense cell-by-gene matrix. Rows show gene z-scores across the 11 Leiden groups; Scatter uses raw counts from the separate `counts` layer.

In [ ]:
cluster_attr = "leiden"
cluster_values = adata.obs[cluster_attr].astype(str)
categories = adata.obs[cluster_attr].cat.categories.astype(str).tolist()
palette = dict(zip(categories, adata.uns["leiden_colors"], strict=True))

signatures = pd.DataFrame(
    {
        category: np.asarray(adata.X[cluster_values.to_numpy() == category].mean(axis=0)).ravel()
        for category in categories
    },
    index=adata.var_names,
)
variable_genes = signatures.var(axis=1).sort_values(ascending=False)
markers = [gene for gene in ["INS", "GCG", "SST", "PPY", "CD3D", "MS4A1"] if gene in adata.var_names]
heatmap_genes = list(dict.fromkeys(markers + variable_genes.index[:80].tolist()))
signatures = signatures.loc[heatmap_genes]
signatures = signatures.loc[signatures.std(axis=1) > 0]
mat = dega.clust.Matrix(
    signatures,
    name="pancreas-scatter-signatures",
    row_entity={"entity": "gene", "attr": "name"},
    col_entity={"entity": "cell", "attr": cluster_attr},
    global_colors=palette,
)
mat.norm(axis="row", by="zscore")
mat.clust()
print(f"Clustergram: {len(signatures):,} genes × {len(categories)} clusters")

## Create and link the three panels

Rerunning this cell closes this example's old widgets and removes its links and callbacks. It leaves unrelated notebook widgets alone. Scatter and Landscape have separate state stores, with explicit links for the state they share.

The same attributes are offered in both cell-color dropdowns: `leiden`, `manual_gate`, and numerical QC columns. New cell labels appear immediately in Scatter. Landscape continues to highlight selected cells; its copy of cell metadata is loaded at construction, so recreate the named Landscape if you want to display newly added annotations there as a category.

In [ ]:
if "close_pancreas_demo" in globals():
    close_pancreas_demo()

base_url = (
    "https://raw.githubusercontent.com/broadinstitute/"
    "celldega_Xenium_human_Pancreas_FFPE/main/"
    "Landscape_Xenium_V1_human_Pancreas_FFPE_outs_webp"
)
cell_attributes = [cluster_attr, "manual_gate", "total_counts", "cell_area", "nucleus_area"]
scatter = dega.viz.Scatter(
    adata, view="genes", x="INS", y="GCG", layer="counts",
    x_scale="log1p", y_scale="log1p", color_by=cluster_attr,
    width=480, height=580, point_size=2, name="pancreas-scatter",
)
clustergram = dega.viz.Clustergram(
    matrix=mat, width=440, height=580, name="pancreas-clustergram",
)
landscape = dega.viz.Landscape(
    technology="Xenium", base_url=base_url, adata=adata,
    cluster_attr=cluster_attr, cell_attr=cell_attributes,
    width=480, height=580, name="pancreas-landscape",
)

# Frontend links keep spatial highlighting responsive without a Python round trip.
links = [
    jsdlink((scatter, "selected_cells"), (landscape, "selected_cells")),
    jsdlink((clustergram, "click_info"), (landscape, "update_trigger")),
]
summary = HTML()

def on_clustergram_click(change):
    event = change["new"] or {}
    kind = event.get("type", "")
    value = event.get("value") or {}
    if kind == "row_label":
        gene = value.get("name") if isinstance(value, dict) else value
        if gene in adata.var_names:
            scatter.set_axes(y=gene)
    elif kind == "col_label" or kind.startswith("col_dendro"):
        names = [value.get("name")] if kind == "col_label" else value.get("selected_names", [])
        names = [str(name) for name in names if name is not None]
        scatter.color_by = cluster_attr
        scatter.selected_categories = names


def on_scatter_axes(change=None):
    clustergram.highlighted_genes = list(dict.fromkeys([scatter.x, scatter.y]))
    if change is not None and change["name"] == "y":
        landscape.trigger_update({"type": "row_label", "value": {"name": scatter.y}})


def update_summary(change=None):
    ids = scatter.selected_cells
    if ids:
        counts = adata.obs.loc[ids, cluster_attr].value_counts()
        text = ", ".join(f"{escape(str(group))}: {count:,}" for group, count in counts.items() if count)
        summary.value = f"<b>{len(ids):,} selected cells</b> — Leiden groups: {text}"
    else:
        summary.value = "No cells selected. Click a Clustergram column, select CELL bars, or use GATE / SKTCH in Scatter."

clustergram.observe(on_clustergram_click, names="click_info")
scatter.observe(on_scatter_axes, names=["x", "y"])
scatter.observe(update_summary, names="selected_cells")
on_scatter_axes()
update_summary()

panels = [
    VBox([HTML("<b>Scatter</b> — genes, UMAP, spatial"), scatter], layout=Layout(min_width="480px")),
    VBox([HTML("<b>Clustergram</b> — gene × Leiden group"), clustergram], layout=Layout(min_width="440px")),
    VBox([HTML("<b>Landscape</b> — tissue context"), landscape], layout=Layout(min_width="480px")),
]
row = HBox(panels, layout=Layout(width="100%", overflow="auto", gap="8px"))
demo = VBox([row, summary])

def close_pancreas_demo():
    for widget_link in links:
        widget_link.unlink()
        widget_link.close()
    clustergram.unobserve(on_clustergram_click, names="click_info")
    scatter.unobserve(on_scatter_axes, names=["x", "y"])
    scatter.unobserve(update_summary, names="selected_cells")
    for widget in [scatter, clustergram, landscape, summary, *[child for panel in panels for child in panel.children if isinstance(child, HTML)], *panels, row, demo]:
        widget.close()

display(demo)

## Try a pancreas gate

1. Zoom into the **INS-high** population. The local CELL bars update as the viewport changes.
2. Use **SKTCH**, click around that population, then click the first vertex to finish. **GATE** also supports dragging a rectangle.
3. Click **LABEL**, choose `manual_gate`, enter `INS-high candidate`, select a color, and Apply. This is an exploratory label, not an automated cell-type call.
4. Switch the view to **UMAP** or **Spatial**. The selected cell IDs and labels remain attached to the same cells; Landscape highlights their tissue locations.
5. Choose `total_counts` from the color dropdown for a numerical gradient, or return to `leiden` and Shift-click several category bars. You can do the same in Landscape's CELL controls.
6. Click another Clustergram gene such as **SST** to change Scatter's Y axis. Change the X dropdown independently.

You can also drive the widget from Python:

In [ ]:
# Optional controls to run individually while exploring:
# scatter.set_axes("INS", "SST", layer="counts")
# scatter.set_scale("log1p")
# scatter.set_view("umap")
# scatter.color_by = "total_counts"
# scatter.color_by = "leiden"
# scatter.selected_categories = ["0", "1"]
scatter.get_view_state()

## Inspect labels and save explicitly

Apply in the LABEL dialog writes into this notebook's in-memory AnnData. Unselected cells retain their prior labels. Nothing is written to the source file automatically. You can also annotate the current selection with `scatter.annotate_selection("manual_gate", "my label", color="#4477aa")`.

In [ ]:
display(adata.obs["manual_gate"].value_counts())
display(adata.obs.loc[scatter.selected_cells, ["leiden", "manual_gate", "total_counts"]].head(20))
# To persist after you are happy with the labels, choose a NEW output path:
# adata.write_h5ad(cache_dir / "pancreas-manually-annotated.h5ad")
# To get an independent AnnData containing only selected cells:
# selected_adata = scatter.get_selection(as_adata=True)

## Cleanup

Run the following when finished to release this example's WebGL contexts and links. Re-running the construction cell also performs this cleanup automatically.

In [ ]:
# close_pancreas_demo()